In [1]:
import sympy as sp
import numpy as np
import scipy as sci

In [2]:
# define independent variables 
x,y = sp.symbols('x, y', real=True)
xv = sp.Matrix([x,y])

# define the rhs of the governing equation
w = sp.symbols("omega")
u1,u2 = sp.symbols("u_x, u_y", real=True)
u1 = sp.Function("u_x")(x, y)
u2 = sp.Function("u_y")(x, y)
u = sp.Matrix([u1,u2])
grad_w = sp.Matrix([sp.Derivative(w,x), sp.Derivative(w,y)])
FU = -u.dot(grad_w)
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [3]:
# define q(t)
N = 2 #number of vortexes

q = sp.Matrix()

A = sp.symbols("A", real=True)
L = sp.symbols("L", real=True, positive=True)

# xc= sp.symbols("x_c", real=True)
# yc= sp.symbols("y_c", real=True)

xc= sp.Matrix()
yc= sp.Matrix()
# r = sp.Matrix()

for i in range(N):
    xc = sp.Matrix([xc, sp.symbols("x_c_"+str(i+1), real=True)])
    yc = sp.Matrix([yc, sp.symbols("y_c_"+str(i+1), real=True)])
    # r = sp.Matrix([r, sp.symbols("r_"+str(i+1), real=True, positive=True)])
    # r = sp.Matrix([r, sp.Function("r_"+str(i+1))(x, y, xc[i], yc[i])])

q = sp.Matrix([A, L, xc, yc])
# qr = sp.Matrix([A, L, r])

q

Matrix([
[    A],
[    L],
[x_c_1],
[x_c_2],
[y_c_1],
[y_c_2]])

In [4]:
# define the ansatz u_hat(x; q)
ansatz_gamma = 0
ansatz_gamma = A*sp.exp(-((x-xc[0])**2+(y-yc[0])**2)/L**2) + A*sp.exp(-((x-xc[1])**2+(y-yc[1])**2)/L**2)

ansatz_gamma

A*exp((-(x - x_c_1)**2 - (y - y_c_1)**2)/L**2) + A*exp((-(x - x_c_2)**2 - (y - y_c_2)**2)/L**2)

In [5]:
ansatz_u = sp.Matrix([
    sp.Derivative(ansatz_gamma,y).doit().simplify(),
    -sp.Derivative(ansatz_gamma, x).doit().simplify()
])

ansatz_u.simplify()
ansatz_u

Matrix([
[-2*A*((y - y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (y - y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**2],
[ 2*A*((x - x_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**2]])

In [6]:
ansatz = (- sp.Derivative(ansatz_gamma, x, 2) - sp.Derivative(ansatz_gamma, y, 2)).doit()
# ansatz = (-sp.Derivative(ansatz_u[0], y) ).doit() + sp.Derivative(ansatz_u[1], x).doit()
ansatz.simplify()

4*A*(L**2*(exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2)) - (x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) - (y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**4

In [7]:
# compute partial derivatives du/dqi
dwdq = ansatz.diff(q)

dwdq.simplify()

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         4*(L**2*(exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2)) - (x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) - (y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**4],
[-8*A*exp(-(x - x_c_1)**2/L**2 - (y - y_c_1)**2/L**2)/L**3 - 8*A*exp(-(x - x_c_2)**2/L**2 - (y 

In [8]:
dwdq[0].simplify()

4*(L**2*(exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2)) - (x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) - (y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**4

In [9]:
# data
a = 1
l = 1
xc1 = 1
yc1 = 1
xc2 = -1
yc2 = -1

def sub_data(f):
    return f.subs(A,a).subs(L,l).subs(xc[0],xc1).subs(yc[0],yc1).subs(xc[1],xc2).subs(yc[1],yc2)

In [10]:
def simplify_exponentials(expr):
    """
    Combines exp(A) * exp(B) -> exp(A + B) and factors exponents.
    """
    # Force combination of exp bases
    expr = sp.powsimp(expr, combine='exp', deep=True)
    # Rewrite terms to group exponents together
    return sp.combine(expr, 'exp')

def safe_exp(val):
    """
    Prevents floating-point overflow in exp().
    Returns 0.0 for large negative values and caps extreme positive values.
    """
    # Cap argument to float64 safe range (~709)
    val = np.clip(val, -700.0, 700.0)
    return np.exp(val)

In [11]:
import scipy.integrate as scipy_integrate

def inner_prod_H(f, g, vars=(x, y), BOUND = 30.*l):
    """
    Computes numerical 2D integral of f * g over R^2 for SymPy expressions.
    """
    # Combine expressions and convert directly to a fast numeric function
    integrand_expr = (sub_data(f*g))
    # Force SymPy to group exponentials together rather than splitting them
    integrand_expr = sp.powsimp(integrand_expr.expand(), combine='exp', deep=True)
    print(integrand_expr)

    custom_modules = [{'exp': safe_exp}, 'numpy']
    integrand_func = sp.lambdify(vars, integrand_expr, modules=custom_modules)

    def integrand(y_val, x_val):
        try:
            val = integrand_func(x_val, y_val)
            return float(val) if np.isfinite(val) else 0.0
        except (OverflowError, FloatingPointError):
            return 0.0

    # # To integrate x first, then y:
    result, error = scipy_integrate.dblquad(
        integrand, 
        -BOUND, BOUND,                    
        -BOUND, BOUND                 
    )

    print(error)
    
    return result

In [12]:
# xmin,xmax = sp.symbols("x_{min}, x_{max}")
# # define the inner product according to the problem
# def inner_prod_H(f, g):
#     ix = sp.integrate((f*g).expand(),(x, -sp.oo, sp.oo))
#     return sp.integrate(ix.expand(),(y, -sp.oo, sp.oo)).expand()

In [13]:
(-4*y**2*sp.exp(-y**2 - (x - 1)**2) - 4*y**2*sp.exp(y**2 - (x + 1)**2) - 4*(x - 1)**2*sp.exp(-y**2 - (x - 1)**2) - 4*(x + 1)**2*sp.exp(y**2 - (x + 1)**2) + 4*sp.exp(-y**2 - (x - 1)**2))**2

(-4*y**2*exp(-y**2 - (x - 1)**2) - 4*y**2*exp(y**2 - (x + 1)**2) - 4*(x - 1)**2*exp(-y**2 - (x - 1)**2) - 4*(x + 1)**2*exp(y**2 - (x + 1)**2) + 4*exp(-y**2 - (x - 1)**2))**2

In [14]:
inner_prod_H(dwdq[0], dwdq[0])
# m00 = (dwdq[0]**2).simplify()

# m00i = sp.integrate(m00.expand(), (x, -sp.oo, +sp.oo))

32*x**4*exp(-2*x**2 - 2*y**2 - 4) + 16*x**4*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) + 16*x**4*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 64*x**3*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) - 64*x**3*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 64*x**2*y**2*exp(-2*x**2 - 2*y**2 - 4) + 32*x**2*y**2*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) + 32*x**2*y**2*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 64*x**2*y*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) - 64*x**2*y*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) - 64*x**2*exp(-2*x**2 - 2*y**2 - 4) + 96*x**2*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) + 96*x**2*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 64*x*y**2*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) - 64*x*y**2*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) - 256*x*y*exp(-2*x**2 - 2*y**2 - 4) + 128*x*y*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) + 128*x*y*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 64*x*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) - 64*x*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 32*y**4*exp(-2*x**2 - 2*y**2 - 4) + 16*y**4*exp(-2*x**2 - 4*x - 2*y**2 

25.59306344134492

In [15]:
# construct the matrix M_ij = <du/dqi, du/dqj>_H
n = len(q)
M = sp.zeros(n, n)

for i in range(n):
    M[i, i] = inner_prod_H(dwdq[i], dwdq[i])
    print(M[i, i])
    for j in range(i+1, n):
        M[i, j] = inner_prod_H(dwdq[i], dwdq[j])
        M[j,i] = M[i,j]
        print(M[i, j])


32*x**4*exp(-2*x**2 - 2*y**2 - 4) + 16*x**4*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) + 16*x**4*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 64*x**3*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) - 64*x**3*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 64*x**2*y**2*exp(-2*x**2 - 2*y**2 - 4) + 32*x**2*y**2*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) + 32*x**2*y**2*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 64*x**2*y*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) - 64*x**2*y*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) - 64*x**2*exp(-2*x**2 - 2*y**2 - 4) + 96*x**2*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) + 96*x**2*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 64*x*y**2*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) - 64*x*y**2*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) - 256*x*y*exp(-2*x**2 - 2*y**2 - 4) + 128*x*y*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) + 128*x*y*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 64*x*exp(-2*x**2 - 4*x - 2*y**2 - 4*y - 4) - 64*x*exp(-2*x**2 + 4*x - 2*y**2 + 4*y - 4) + 32*y**4*exp(-2*x**2 - 2*y**2 - 4) + 16*y**4*exp(-2*x**2 - 4*x - 2*y**2 

In [16]:
M

Matrix([
[  25.5930634413449, -27.4343522918663,     0.460322212625829,    -0.460322212625829,     0.460322212624969,    -0.460322212624969],
[ -27.4343522918663,  95.0070983633252,      2.76193327577381,     -2.76193327577381,      2.76193327577434,     -2.76193327577434],
[ 0.460322212625829,  2.76193327577381,        37.69911184307,       1.6111277441997, -3.13818942015462e-10,      1.84128885051426],
[-0.460322212625829, -2.76193327577381,       1.6111277441997,        37.69911184307,      1.84128885051426, -3.13818942015462e-10],
[ 0.460322212624969,  2.76193327577434, -3.13818942015462e-10,      1.84128885051426,      37.6991118430621,      1.61112774421155],
[-0.460322212624969, -2.76193327577434,      1.84128885051426, -3.13818942015462e-10,      1.61112774421155,      37.6991118430621]])

In [17]:
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [18]:
# compute rhs from the ansatz
Fua = FU.subs(u1, ansatz_u[0]).subs(u2, ansatz_u[1]).subs(w, ansatz).doit()
Fua.simplify()

16*A**2*(((x - x_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))*(2*L**2*((y - y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (y - y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2)) + (x - x_c_1)**2*(-y + y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)**2*(-y + y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + (-y + y_c_1)**3*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (-y + y_c_2)**3*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2)) - ((y - y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (y - y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))*(2*L**2*((x - x_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2)) + (-x + x_c_1)**3*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (-x + x_c_1)*(y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (-x + x_c_2)**3*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + (-x + x_c_2)*(y - y_c

In [20]:
# compute f
n = len(q)
f = sp.zeros(n, 1)

for i in range(n):
    f[i] = inner_prod_H(dwdq[i], Fua)
    print(f[i])

f

512*x**4*exp(-3*x**2 - 2*x - 3*y**2 - 2*y - 6) + 512*x**4*exp(-3*x**2 + 2*x - 3*y**2 + 2*y - 6) + 1024*x**3*exp(-3*x**2 - 2*x - 3*y**2 - 2*y - 6) - 1024*x**3*exp(-3*x**2 + 2*x - 3*y**2 + 2*y - 6) + 1024*x**2*y*exp(-3*x**2 - 2*x - 3*y**2 - 2*y - 6) - 1024*x**2*y*exp(-3*x**2 + 2*x - 3*y**2 + 2*y - 6) + 512*x**2*exp(-3*x**2 - 2*x - 3*y**2 - 2*y - 6) + 512*x**2*exp(-3*x**2 + 2*x - 3*y**2 + 2*y - 6) - 1024*x*y**2*exp(-3*x**2 - 2*x - 3*y**2 - 2*y - 6) + 1024*x*y**2*exp(-3*x**2 + 2*x - 3*y**2 + 2*y - 6) - 512*y**4*exp(-3*x**2 - 2*x - 3*y**2 - 2*y - 6) - 512*y**4*exp(-3*x**2 + 2*x - 3*y**2 + 2*y - 6) - 1024*y**3*exp(-3*x**2 - 2*x - 3*y**2 - 2*y - 6) + 1024*y**3*exp(-3*x**2 + 2*x - 3*y**2 + 2*y - 6) - 512*y**2*exp(-3*x**2 - 2*x - 3*y**2 - 2*y - 6) - 512*y**2*exp(-3*x**2 + 2*x - 3*y**2 + 2*y - 6)
1.463084635814683e-08
1.17700987845026e-12
1024*x**6*exp(-3*x**2 - 2*x - 3*y**2 - 2*y - 6) + 1024*x**6*exp(-3*x**2 + 2*x - 3*y**2 + 2*y - 6) + 4096*x**5*exp(-3*x**2 - 2*x - 3*y**2 - 2*y - 6) - 4096*x**5

Matrix([
[1.17700987845026e-12],
[7.38858974003165e-12],
[  -0.639155188734293],
[   0.639155188734293],
[   0.639155188722568],
[  -0.639155188722568]])

In [21]:
q_dot = M.inv()*f

q_dot.simplify()

In [22]:
q_dot

Matrix([
[2.40834912776378e-13],
[ 1.6842148335694e-13],
[  -0.016851237554255],
[   0.016851237554255],
[  0.0168512375538887],
[ -0.0168512375538887]])